# Capa Silver - Transacciones de Venta

## Librerias

In [1]:
from pyspark.sql import functions as f

## Variables

In [2]:
%run ../00-Modulos/n_modulo_variables.ipynb

Servicios:
 Warehouse  = s3a://iceberg/warehouse
 EndPoint   = http://localhost:9000
 Uri        = thrift://localhost:9083
Base de Datos Landing =  iceberg.landing
Base de Datos Bronze  =  iceberg.bronze
Base de Datos Silver  =  iceberg.silver
Base de Datos Gold    =  iceberg.gold


## Spark Session

In [3]:
spark = spark_session_hive("Lakehouse-Iceberg")

## Extracción

In [4]:
df_sales_transactions = spark.read.table(f'{vSchemaLand}.sales_transactions')
df_sales_transaction = spark.read.table(f'{vSchemaBron}.sales_transaction')
df_products = spark.read.table(f'{vSchemaSilv}.products')

## Transformaciones

In [5]:
# Seleccion de columnas finales
df_sales_transactions = df_sales_transactions \
    .select(f.date_format(f.col("TRANSACTION_DATE"), "yyyy-MM-dd").alias("TRANSACTION_DATE")) \
    .distinct()
df_source = df_sales_transaction.alias("bt") \
    .join(df_sales_transactions.alias("lt"), (f.date_format(f.col("bt.TRANSACTION_DATE"), "yyyy-MM-dd") == f.col("lt.TRANSACTION_DATE")), "inner") \
    .join(df_products.alias("p"), f.col("bt.PRODUCT") == f.col("p.PRODUCT_ID"), "inner") \
    .select(
        f.col("bt.TRANSACTION_DATE_ID"),
        f.col("bt.CUST_NO").alias("CUST_ID"),
        f.col("bt.PRODUCT").alias("PRODUCT_ID"),
        f.col("bt.DEALERSHIP").alias("DEALERSHIP_ID"),
        f.col("bt.PAYMENT_DESC"),
        f.col("bt.PROMO_ID"),
        f.col("bt.DATE_ID"),
        f.col("bt.TRANSACTION_DATE"),
        f.col("bt.TRANSACTION_ID"),
        f.col("bt.EMPLOYEE_ID"),
        f.col("bt.TIME_KEY"),
        f.col("bt.SELLING_PRICE"),
        f.col("p.PRODUCT_COST").alias("UNIT_COST"),
        f.col("bt.DELIVERY_CHARGES"),
        f.col("bt.QUANTITY").alias("SALES_QTY"),
        f.col("bt.DISCOUNT"),
        f.col("bt.HOLDBACK"),
        f.col("bt.REBATE")
    )

## Carga

In [6]:
# Obtenemos variables para merge
target_table = f'{vSchemaSilv}.transactions'
join_columns = ["CUST_ID", "PRODUCT_ID", "DEALERSHIP_ID", "PAYMENT_DESC", "PROMO_ID", "TIME_KEY"]

# Realiza el merge de la información
iceberg_upsert(df_source, target_table, join_columns, "TRANSACTION_DATE_ID")